In [6]:
import pandas as pd
import numpy as np

In [15]:
df_freq = pd.read_csv("../../data/freMTPL2freq.csv")
df_sev = pd.read_csv("../../data/df_sev_clean.csv")

# df_fre.columns
# df_sev.columns

claims_per_policy = df_sev.groupby('IDpol')['ClaimAmount'].sum().reset_index()

# print(claims_per_policy)

df_merged = pd.merge(df_freq, claims_per_policy, on='IDpol', how='left')
df_merged['ClaimAmount'] = df_merged['ClaimAmount'].fillna(0)

# چند بیمه‌نامه منحصربه‌فرد داریم؟
policy_count = df_merged['IDpol'].nunique()
# مجموع مدت زمانی که بیمه‌نامه‌ها تحت پوشش بوده‌اند.
total_exposure = df_merged['Exposure'].sum()
# در مجموع چند خسارت ثبت شده است؟
total_claims = df_merged['ClaimNb'].sum()
# مجموع هزینه مالی تمام خسارت‌ها چقدر بوده است؟
total_claim_cost = df_merged['ClaimAmount'].sum()

# چند بار خسارت اتفاق می‌افتد؟
claim_frequency = total_claims / total_exposure
# هر خسارت به طور متوسط چقدر هزینه داشته؟
average_severity = total_claim_cost / total_claims if total_claims > 0 else 0  

kpi_summary = pd.DataFrame({
    'KPI': [
        'Policy Count', 
        'Total Exposure (Years)', 
        'Total Claims', 
        'Claim Frequency (Claims/Year)', 
        'Total Claim Cost', 
        'Average Claim Severity'
    ],
    'Value': [
        policy_count, 
        round(total_exposure, 2), 
        total_claims, 
        round(claim_frequency, 4), 
        round(total_claim_cost, 2), 
        round(average_severity, 2)
    ]
})

pd.set_option('display.float_format', '{:,.4f}'.format)

# جواب: چرا فقط شمردن تعداد خسارت‌ها می‌تواند گمراه‌کننده باشد؟
# مثلاً فرض کنید شرکت الف فقط ۱۰۰ بیمه‌نامه دارد و ۲۰ خسارت ثبت کرده،
# در حالی که شرکت ب دارای ۱۰٬۰۰۰ بیمه‌نامه است و ۱۰۰ خسارت ثبت کرده است.
# اگر فقط تعداد خام خسارت‌ها را بررسی کنیم، به نظر می‌رسد شرکت ب وضعیت بدتری دارد،
# اما در واقع نسبت خسارت در شرکت الف ۲۰٪ و در شرکت ب فقط ۱٪ است.
# بنابراین تعداد خام خسارت‌ها به‌تنهایی می‌تواند گمراه‌کننده باشد.

print(kpi_summary)

                             KPI           Value
0                   Policy Count    677,991.0000
1         Total Exposure (Years)    358,482.8400
2                   Total Claims     26,444.0000
3  Claim Frequency (Claims/Year)          0.0738
4               Total Claim Cost 59,909,216.5000
5         Average Claim Severity      2,265.5100
